<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

# Notebook 03 – Data Mining: Association Rules & Clustering
**Đề 2: Dự đoán Bệnh Tim**

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import sys
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.loader import DataLoader, load_config
from src.features.builder import FeatureBuilder
from src.mining.association import AssociationMiner
from src.mining.clustering import ClusteringMiner
from src.visualization.plots import Plotter
from src.evaluation.report import Reporter

cfg = load_config('../configs/params.yaml')
loader = DataLoader('../configs/params.yaml')
plotter = Plotter(cfg)
reporter = Reporter(cfg)
print("Modules OK")

FileNotFoundError: [Errno 2] No such file or directory: '../configs/params.yaml'

## PHẦN 1: Luật kết hợp (Association Rules – Apriori)

### 1.1 Chuẩn bị transaction data

In [ ]:
df_raw = loader.load_raw()
builder = FeatureBuilder(cfg)
df_onehot = builder.build_onehot_transactions(df_raw)
print(f"Transaction matrix: {df_onehot.shape}")
print(f"Số items: {df_onehot.columns.tolist()[:10]}...")

### 1.2 Chạy Apriori

In [ ]:
miner = AssociationMiner(cfg)
frequent_items = miner.run_apriori(df_onehot)
print(f"\nTop 10 frequent itemsets theo support:")
print(frequent_items.nlargest(10, 'support')[['itemsets','support']].to_string())

### 1.3 Tạo và phân tích luật

In [ ]:
rules = miner.generate_rules()
print(f"\nTổng số luật: {len(rules)}")
print("\nTop 10 luật theo Lift:")
print(miner.format_rules(miner.get_top_rules(10)).to_string())

In [ ]:
print("\nLuật dẫn đến BỆNH TIM (consequent=disease):")
disease_rules = miner.get_disease_rules()
print(miner.format_rules(disease_rules.head(15)).to_string())

In [ ]:
# Thống kê tóm tắt
print("\nThống kê luật kết hợp:")
print(miner.rules_summary().to_string())
reporter.summarize_association_rules(miner.format_rules(disease_rules.head(20)))

### 1.4 Trực quan hóa luật

In [ ]:
plotter.plot_rules_scatter(rules)
formatted_top = miner.format_rules(miner.get_top_rules(15))
plotter.plot_top_rules_lift(formatted_top)

### 1.5 Diễn giải luật quan trọng
**Ví dụ luật:** `{cp=0, exang=1, thal=2} → {disease}` với lift > 2

**Ý nghĩa:** Bệnh nhân có đau ngực dạng điển hình (cp=0), đau khi gắng sức (exang=1) và thalassemia reversible defect (thal=2) có nguy cơ bệnh tim cao gấp 2 lần so với trung bình.

**Khuyến nghị:** Sử dụng các tổ hợp triệu chứng này để ưu tiên tầm soát sớm.

## PHẦN 2: Phân cụm (Clustering)

### 2.1 Chuẩn bị dữ liệu cho clustering

In [ ]:
df_proc = pd.read_parquet('../data/processed/heart_processed.parquet')
from src.features.builder import FeatureBuilder
builder2 = FeatureBuilder(cfg)
X, y = builder2.get_X_y(df_proc)
X_arr = X.values
print(f"X shape: {X_arr.shape}")

### 2.2 Chọn K tối ưu (Elbow + Silhouette)

In [ ]:
clusterer = ClusteringMiner(cfg)
k_results = clusterer.find_optimal_k(X_arr)
plotter.plot_elbow(k_results)

### 2.3 KMeans với K tốt nhất

In [ ]:
k_best = 3
labels_km = clusterer.fit_kmeans(X_arr, k=k_best)
km_eval = clusterer.evaluate_all(X_arr, labels_km)
print("Đánh giá KMeans:", km_eval)

### 2.4 HAC (Hierarchical Agglomerative Clustering)

In [ ]:
labels_hac = clusterer.fit_hac(X_arr, k=k_best)
hac_eval = clusterer.evaluate_all(X_arr, labels_hac)
print("Đánh giá HAC:", hac_eval)

### 2.5 DBSCAN

In [ ]:
labels_db = clusterer.fit_dbscan(X_arr)
db_eval = clusterer.evaluate_all(X_arr, labels_db)
print("Đánh giá DBSCAN:", db_eval)

### 2.6 Bảng so sánh thuật toán phân cụm

In [ ]:
from src.evaluation.metrics import Metrics
clust_results = [
    Metrics.clustering_summary(X_arr, labels_km, "KMeans"),
    Metrics.clustering_summary(X_arr, labels_hac, "HAC"),
    Metrics.clustering_summary(X_arr, labels_db, "DBSCAN"),
]
clust_df = reporter.summarize_clustering(clust_results)
print(clust_df.to_string())

### 2.7 Trực quan hóa phân cụm (PCA 2D)

In [ ]:
X_2d = clusterer.fit_pca(X_arr, n_components=2)
plotter.plot_clusters_2d(X_2d, labels_km, f"KMeans K={k_best} (PCA 2D)")

### 2.8 Hồ sơ (Profiling) các cụm

In [ ]:
df_for_profile = df_proc.copy()
profile = clusterer.profile_clusters(df_for_profile, labels_km)
print("Hồ sơ các cụm KMeans:")
print(profile.to_string())
plotter.plot_cluster_profiles(profile)

In [ ]:
cluster_names = clusterer.name_clusters(profile)
print("Tên cụm:", cluster_names)
reporter.save_table(profile, 'cluster_profiles.csv', index=True)

### 2.9 Insight phân cụm
- **Cụm nguy cơ cao:** Tuổi cao, thalach thấp, oldpeak cao, exang=1
- **Cụm nguy cơ trung bình:** Có một số triệu chứng nhưng chưa rõ ràng  
- **Cụm nguy cơ thấp:** Trẻ hơn, các chỉ số trong ngưỡng bình thường

**Ứng dụng:** Ưu tiên tầm soát và can thiệp y tế theo nhóm nguy cơ.

In [ ]:
print("Notebook 03 hoàn tất.")